# 线性代数与微积分的最小补充

---
## 标量, 向量, 矩阵, 张量与形状
### 四种数据结构
- **标量 (scalar)**: 单独一个数, 例如5.0
- **向量 (vector)**: 一列或一行数, 例如[2, 5, 7\]
- **矩阵 (matrix)**: 按行和列排成的数字表, 例如[[1, 2, 3\], [4, 5, 6\]\]
- **张量 (tensor)**: 能表示任意轴数数据的通用结构. 向量, 矩阵都是张量的特例. 在PyTorch里, 它们都由torch.Tensor表示

可以把它们想成数据排列形式逐渐丰富: 

<table>
<tr><th>数据</th><th>PyTorch形状</th><th>轴数</th></tr>
<tr><td>一个数5.0</td><td>()</td><td>0</td></tr>
<tr><td>向量[2, 5, 7]</td><td>(3, )</td><td>1</td></tr>
<tr><td>2行3列的矩阵</td><td>(2, 3)</td><td>2</td></tr>
<tr><td>RGB图片, 3个颜色通道</td><td>(3, H, W)</td><td>3</td></tr>
</table>

**轴 (axis)** 就是数据排列的一个方向; **形状 (shape)** 则一次记录每个轴有多长. 比如 (2, 3) 表示一个轴长度为2, 第二个轴长度为3, 也就是2行3列. 

这里容易被"维度"这个词绕晕: 
- **张量的阶数/维数 (rank/number of dimensions)** 有几个轴. (2, 3)有2个轴. 
- **某个轴的长度**: 该方向上有多少个元素. (2, 3)的两个轴长度分别是2和3. 

所以, (3, )是一个轴, 长度为3; (3, 1)则是两个轴, 长度分别为3和1. 它们不是同一种形状. 注意, 线性代数里"matrix rank"还有另一个专门含义; 我们这里说张量的rank时, 指的是轴数.

把它放回熟悉的图片例子里: 
```
[64, 1, 28, 28]
```

按常见的图片批次约定, 它表示64张图片, 每张1个灰度通道, 高28, 宽28. 它有4个轴; 但形状数字本身只记录长度, 轴具体代表什么还要看代码采用的约定. PyTorch中可以用x.shape查看形状, 用x.ndim查看轴数. 

这也是为什么读模型代码时要盯住shape: 不仅要知道数据有多少个值, 还要知道它们按什么结构排列.

---
## 点积与矩阵乘法
先分清两个运算: 

**逐元素乘法(element-wise multiplication)**: 对应位置的数字分别相乘. 比如: 
```
[1, 2, 3] * [4, 5, 6] = [4, 10, 18]
```

在PyTorch里, x * y和torch.mul(x, y)都是逐元素乘法. 这里用相同形状的向量举例, 结果仍是一个向量. 

**点积 (dot production)**: 先把对应位置相乘, 再把乘积加起来: 
```
[1, 2, 3] · [4, 5, 6]
= 1 x 4 + 2 x 5 + 3 x 6
= 32
```

点积把两个向量合成一个数. 两边向量长度必须相同. PyTorch的torch.dot(x, y)专门处理一维向量; 一维向量也可以写成 x @ y 或 torch.matmul(x, y), 结果是形状为()的标量张量

In [ ]:
import torch

x = torch.tensor([1., 2., 3.])
y = torch.tensor([4., 5., 6.])

x * y           # tensor([4., 10., 18.])
torch.dot(x, y) # tensor(32.)
x @ y            # tensor(32.)

### 矩阵乘法 (matrix multiplication)
可以理解成: 左边矩阵的每一行, 分别和右边矩阵的每一列做点积. 

形状规则是: 
```
(m, n) @ (n, p) -> (m, p)
```

两个矩阵中间相邻的n必须相同; 输出保留外侧的m和p. 例如: 

In [ ]:
A = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.],
])  # shape: (2, 3)

B = torch.tensor([
    [1., 2.],
    [3., 4.],
    [5., 6.],
])  # shape: (3, 2)

A @ B
# tensor([[22., 28.],
#         [49., 64.]])

PyTorch中的 @ 和 torch.matmul(A, B) 都能做矩阵乘法. 

---
## 范数

**范数 (Norm)** 可以先理解成: 用一个数字表示向量"有多大". 它衡量的是向量分量的大小, 不是有几个分量. 比如一个向量有3个分量, 并不意味着它的范数是3. 

先认识两个会用到的词: 
- **绝对值 (absolute value)**: 不看正负号, 只看离0多远. 
- **平方根 (square root)**: 哪个数乘以自己等于原数. 

### L1范数和L2范数

拿向量 v = [-3, 4\]举例: 
- L1范数: 把每个分量的绝对值加起来. |-3|+|4| = 3 + 4 = 7.
- L2范数: 把每个分量平方, 相加, 再开平方. $\sqrt{(-3)^2 + 4^2} = 5$

L2范数也可以想象成: 从坐标原点走到点 (-3, 4)的直线距离. L1范数则像只能沿着横向和纵向走时, 两段路程加起来. 

注意: L1和L2都把整个向量概括成一个数, 所以结果的形状是 torch.size([]), 和刚才的点积一样是标量张量; 但它们在计算的东西不同. 

In [ ]:
import torch

v = torch.tensor([-3.0, 4.0])

print(torch.linalg.vector_norm(v, ord=1))  # tensor(7.)
print(torch.linalg.vector_norm(v, ord=2))  # tensor(5.)

ord=1选择L1范数, ord=2选择L2范数. 这里用torch.linalg.vector_norm明确表示"计算向量的范数". 

---
## 导数, 偏导数, 梯度, 链式法则
### 1. 导数: 输入变一点, 输出变多块? 
**函数 (function)** 就是一种输入到输出的规则. 例如$f(x) = x^2$: 输入$x$, 输出它的平方. 
**导数 (derivative)** 描述输入在某个位置附近稍微变化时, 输出大约变化多块. 它也可以理解为曲线在该位置的斜率. 


### 2. 偏导数: 一次只看一个输入
当函数有多个输入时, 比如$f(x, y)$, **偏导数 (partial derivative)** 就是暂时固定其他输入, 只观察其中一个输入变化时, 输出如何变化. 


## 3. 梯度: 把所有偏导数组成一个向量
**梯度 (gradient)** 是函数对每个输入的偏导数排成的向量. 梯度指向函数值上升最快的方向. 

深度学习中的模型通常有很多参数, 损失函数会依赖这些参数. 梯度能告诉我们损失对各参数分别有多敏感; 之后讲训练会用到它. 


### 4. 链式法则: 变化随着函数链传递

**复合函数 (composite function)** 是把一个函数接在另一个函数后面: 前一个函数的输出, 成为后一个函数的输入. 

**链式法则 (chain rule)** 就是把两段变化率相乘:
```
dy/dx = (dy/du) x (du/dx)
```

神经网络可以看作许多函数前后连起来. 链式法则让我们能沿着这条连接关系, 逐层计算参数对最终损失的影响.